# Python Intern Interview L1 — Complete Solution (Q1–Q7)
Fully self-contained: every question's complete implementation is inline in this file (no imports from `src/`). Run top-to-bottom in a fresh runtime. The repo also keeps modular `src/` sources plus the formal `tests/` suite (`python -m pytest tests/ -v`).

In [1]:
!pip install -q opencv-python-headless numpy pillow

## Question 1 — Voice-record dedup + quality classification
Strategy: group by document `id`, keep the **highest quality** record (tie → earliest `time`); input is never mutated; every removal is logged.

In [2]:
"""Question 1 — document de-duplication and quality classification.

Duplicate strategy
------------------
Several scans can share the same document ``id``.  We keep the record with the
**highest quality** score because a later re-capture of the same page is almost
always an attempt to repair a poor scan, so the best available quality is the
strongest evidence of content fidelity.  Ties on quality are broken by the
**earliest timestamp**, which makes the outcome fully deterministic and
reproducible.  Every discarded record is written to the logger with its id and
the reason, so the decision is auditable.  The caller's ``documents`` list is
never mutated: all work happens on a ``copy.deepcopy`` of the input.
"""

from __future__ import annotations

import copy
import logging
from collections.abc import Iterable
from typing import Any

logger = logging.getLogger(__name__)

ACCEPT_THRESHOLD = 0.80
REVIEW_THRESHOLD = 0.50

VALID_STRATEGIES = ("keep_best", "keep_first")


def classify_quality(quality: float) -> str:
    """Classify a quality score.

    ``>= 0.80`` -> ``ACCEPT``; ``0.50 <= q < 0.80`` -> ``REVIEW``;
    ``< 0.50`` -> ``RETAKE``.
    """
    if quality >= ACCEPT_THRESHOLD:
        return "ACCEPT"
    if quality >= REVIEW_THRESHOLD:
        return "REVIEW"
    return "RETAKE"


def deduplicate_documents(
    documents: Iterable[dict[str, Any]], strategy: str = "keep_best"
) -> list[dict[str, Any]]:
    """Return one record per document id, without mutating ``documents``.

    ``strategy="keep_best"`` (default) keeps the highest quality record and
    breaks ties on the earliest ``time``.  ``strategy="keep_first"`` keeps the
    first record seen for each id.  Output preserves first-appearance order.
    """
    if strategy not in VALID_STRATEGIES:
        raise ValueError(
            f"unknown strategy {strategy!r}; expected one of {VALID_STRATEGIES}"
        )

    working = copy.deepcopy(list(documents))
    best: dict[Any, dict[str, Any]] = {}
    order: list[Any] = []

    for record in working:
        doc_id = record.get("id")
        if doc_id not in best:
            best[doc_id] = record
            order.append(doc_id)
            continue

        if strategy == "keep_first":
            logger.info("duplicate removed: id=%s (keep_first)", doc_id)
            continue

        current = best[doc_id]
        new_quality = record.get("quality", float("-inf"))
        cur_quality = current.get("quality", float("-inf"))
        new_time = record.get("time", float("inf"))
        cur_time = current.get("time", float("inf"))

        if new_quality > cur_quality or (
            new_quality == cur_quality and new_time < cur_time
        ):
            logger.info(
                "duplicate removed: id=%s (replaced: quality=%s time=%s)",
                doc_id,
                new_quality,
                new_time,
            )
            best[doc_id] = record
        else:
            logger.info("duplicate removed: id=%s (kept existing)", doc_id)

    return [best[doc_id] for doc_id in order]


def process_documents(documents: list[dict[str, Any]]) -> dict[str, Any]:
    """Deduplicate, classify and summarise ``documents``.

    Returns a dict with ``unique_documents``, ``classified`` (each unique record
    augmented with a ``status``), per-category ``counts``, ``average_quality``
    (``0.0`` for an empty input) and ``duplicates_removed``.
    """
    unique_documents = deduplicate_documents(documents, strategy="keep_best")

    classified: list[dict[str, Any]] = []
    counts = {"ACCEPT": 0, "REVIEW": 0, "RETAKE": 0}
    total_quality = 0.0

    for record in unique_documents:
        status = classify_quality(record.get("quality", 0.0))
        counts[status] += 1
        total_quality += record.get("quality", 0.0)
        enriched = dict(record)
        enriched["status"] = status
        classified.append(enriched)

    average_quality = total_quality / len(unique_documents) if unique_documents else 0.0

    return {
        "unique_documents": unique_documents,
        "classified": classified,
        "counts": counts,
        "average_quality": average_quality,
        "duplicates_removed": len(list(documents)) - len(unique_documents),
    }


if __name__ == "__main__":  # pragma: no cover - demo output
    sample_documents = [
        {"id": "D101", "quality": 0.92, "time": 101},
        {"id": "D102", "quality": 0.43, "time": 102},
        {"id": "D103", "quality": 0.81, "time": 103},
        {"id": "D101", "quality": 0.92, "time": 104},
        {"id": "D104", "quality": 0.28, "time": 105},
        {"id": "D105", "quality": 0.74, "time": 105},
        {"id": "D102", "quality": 0.92, "time": 106},
    ]

    result = process_documents(sample_documents)
    print("Unique documents:", len(result["unique_documents"]))
    for record in result["classified"]:
        print(f"  {record['id']}: quality={record['quality']} -> {record['status']}")
    print("Counts:", result["counts"])
    print(f"Average quality: {result['average_quality']:.3f}")
    print("Duplicates removed:", result["duplicates_removed"])


Unique documents: 5
  D101: quality=0.92 -> ACCEPT
  D102: quality=0.92 -> ACCEPT
  D103: quality=0.81 -> ACCEPT
  D104: quality=0.28 -> RETAKE
  D105: quality=0.74 -> REVIEW
Counts: {'ACCEPT': 3, 'REVIEW': 1, 'RETAKE': 1}
Average quality: 0.734
Duplicates removed: 2


In [3]:
documents = [
    {"id": "D101", "quality": 0.92, "time": 101},
    {"id": "D102", "quality": 0.43, "time": 102},
    {"id": "D103", "quality": 0.81, "time": 103},
    {"id": "D101", "quality": 0.92, "time": 104},
    {"id": "D104", "quality": 0.28, "time": 105},
    {"id": "D105", "quality": 0.74, "time": 105},
    {"id": "D102", "quality": 0.92, "time": 106},
]
result = process_documents(documents)
print("counts:", result["counts"])
print("average_quality:", round(result["average_quality"], 4))
print("duplicates_removed:", result["duplicates_removed"])
for c in result["classified"]:
    print(c)
print("input untouched:", len(documents) == 7 and "status" not in documents[0])


counts: {'ACCEPT': 3, 'REVIEW': 1, 'RETAKE': 1}
average_quality: 0.734
duplicates_removed: 2
{'id': 'D101', 'quality': 0.92, 'time': 101, 'status': 'ACCEPT'}
{'id': 'D102', 'quality': 0.92, 'time': 106, 'status': 'ACCEPT'}
{'id': 'D103', 'quality': 0.81, 'time': 103, 'status': 'ACCEPT'}
{'id': 'D104', 'quality': 0.28, 'time': 105, 'status': 'RETAKE'}
{'id': 'D105', 'quality': 0.74, 'time': 105, 'status': 'REVIEW'}
input untouched: True


## Question 2 — Document image quality gate for OCR
Checks: blur = variance of Laplacian (threshold 100), brightness = mean grayscale intensity (usable band 60–200). Tested on 3 synthetic cases.

In [4]:
"""Question 2 - Image quality gate for OCR preprocessing.

Decides whether a document image is sharp enough and correctly exposed to be
fed into an OCR pipeline.  Everything works on in-memory ``numpy`` arrays so the
module has no dependency on cameras, files or tesseract.

Threshold rationale
-------------------
* **Blur** - ``cv2.Laplacian(gray, cv2.CV_64F).var()`` is the standard
  "focus measure" used across the OCR literature (Pezeshki et al., document
  image quality assessment).  Sharp document text typically scores
  ``> 100-150``; visibly blurred scans fall ``< 50``.  ``100.0`` is therefore a
  sensible default cutoff for document OCR.
* **Brightness** - keeping the mean grayscale in ``(60, 200)`` preserves enough
  separation between dark text and light background for adaptive/Otsu
  thresholding.  Below ~60 the ink and paper collapse together; above ~200 the
  background saturates and thin strokes are lost.
* All three thresholds are *tunable* and should be re-calibrated on a labelled
  validation set for each camera / lighting setup.
"""

from __future__ import annotations

import cv2
import numpy as np

# Standard document-OCR defaults; see module docstring for justification.
DEFAULT_BLUR_THRESHOLD = 100.0
DEFAULT_DARK_THRESHOLD = 60.0
DEFAULT_BRIGHT_THRESHOLD = 200.0

_IMAGE_SHAPE = (480, 640)  # (height, width)


def _to_gray(image: np.ndarray) -> np.ndarray:
    """Return a single-channel uint8 grayscale view of ``image``.

    Accepts a 2-D grayscale array, a 3-channel BGR array (OpenCV default) or a
    3-channel RGB array.  Colour arrays are collapsed with ``cv2.cvtColor``;
    because only luminance statistics are computed, the minor BGR/RGB channel
    ordering difference does not change the outcome.
    """
    if image.ndim == 2:
        gray = image
    elif image.ndim == 3 and image.shape[2] == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        raise ValueError(
            f"Unsupported image shape {image.shape}; expected (H, W) or (H, W, 3)"
        )
    if gray.dtype != np.uint8:
        gray = np.clip(gray, 0, 255).astype(np.uint8)
    return gray


def blur_score_laplacian(gray: np.ndarray) -> float:
    """Focus measure via variance of the Laplacian (higher = sharper)."""
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())


def brightness_mean(gray: np.ndarray) -> float:
    """Mean grayscale intensity in the 0-255 range."""
    return float(np.mean(gray))


def contrast_std(gray: np.ndarray) -> float:
    """Standard deviation of grayscale intensity (extra diagnostic)."""
    return float(np.std(gray))


def check_image_quality(
    image: np.ndarray,
    blur_threshold: float = DEFAULT_BLUR_THRESHOLD,
    dark_threshold: float = DEFAULT_DARK_THRESHOLD,
    bright_threshold: float = DEFAULT_BRIGHT_THRESHOLD,
) -> dict[str, object]:
    """Assess whether ``image`` is suitable for OCR.

    Args:
        image: BGR, RGB or grayscale ``numpy`` array.
        blur_threshold: Minimum Laplacian variance considered sharp.
        dark_threshold: Brightness below this is treated as underexposed.
        bright_threshold: Brightness at/above this is treated as overexposed.

    Returns:
        dict with keys ``is_usable`` (bool), ``blur_score``, ``brightness``,
        ``contrast`` (floats, rounded to 1 decimal) and ``reason`` (str).
    """
    gray = _to_gray(image)

    blur_score = blur_score_laplacian(gray)
    brightness = brightness_mean(gray)
    contrast = contrast_std(gray)

    is_sharp = blur_score >= blur_threshold
    is_exposed = dark_threshold < brightness < bright_threshold
    is_usable = is_sharp and is_exposed

    return {
        "is_usable": bool(is_usable),
        "blur_score": round(blur_score, 1),
        "brightness": round(brightness, 1),
        "contrast": round(contrast, 1),
        "reason": _build_reason(blur_score, brightness, blur_threshold,
                                dark_threshold, bright_threshold),
    }


def _build_reason(
    blur_score: float,
    brightness: float,
    blur_threshold: float,
    dark_threshold: float,
    bright_threshold: float,
) -> str:
    """Compose a human-readable verdict from the raw measurements."""
    issues: list[str] = []
    if blur_score < blur_threshold:
        issues.append("blurred")
    if brightness <= dark_threshold:
        issues.append("underexposed")
    if brightness >= bright_threshold:
        issues.append("overexposed")

    if not issues:
        return "Image suitable for OCR"
    if set(issues) == {"blurred", "underexposed"}:
        return "Image may be blurred or underexposed"
    return "Image is " + " and ".join(issues)


# ---------------------------------------------------------------------------
# Deterministic synthetic image generators (no files required).
# ---------------------------------------------------------------------------

def _blank_canvas() -> np.ndarray:
    """White 3-channel canvas matching ``_IMAGE_SHAPE``."""
    return np.full((*_IMAGE_SHAPE, 3), 255, dtype=np.uint8)


def _draw_text_bars(image: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Paint black bars + text onto ``image`` to emulate a text document.

    Filled rectangles carry most of the ink (realistic ~35-45% page coverage);
    a rendered text line adds genuine glyph edges so the Laplacian focus
    measure reacts to text sharpness rather than only to block edges.
    """
    h, w = image.shape[:2]
    margin = 48
    bar_h = 30
    gap = 10
    usable = w - 2 * margin
    y = margin
    row = 0
    while y + bar_h <= h - margin:
        if row == 1:
            cv2.putText(
                image,
                "Document OCR sample line",
                (margin, y + bar_h - 6),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 0, 0),
                2,
                cv2.LINE_AA,
            )
        else:
            # Uneven line lengths look like natural justified paragraphs.
            line_w = int(usable * rng.uniform(0.75, 0.95))
            cv2.rectangle(image, (margin, y), (margin + line_w, y + bar_h),
                          (0, 0, 0), thickness=-1)
        y += bar_h + gap
        row += 1
    return image


def make_clear_image() -> np.ndarray:
    """Sharply rendered, well-exposed document image (should be usable)."""
    rng = np.random.default_rng(42)
    return _draw_text_bars(_blank_canvas(), rng)


def make_blurred_image() -> np.ndarray:
    """Clear image degraded with a 15x15 Gaussian blur (should be unusable)."""
    clear = make_clear_image()
    return cv2.GaussianBlur(clear, (15, 15), 0)


def make_dark_image() -> np.ndarray:
    """Clear image scaled to 25% brightness (should be unusable)."""
    clear = make_clear_image().astype(np.float32) * 0.25
    return np.clip(clear, 0, 255).astype(np.uint8)


if __name__ == "__main__":  # pragma: no cover - manual demo
    for name, factory in (
        ("clear", make_clear_image),
        ("blurred", make_blurred_image),
        ("dark", make_dark_image),
    ):
        result = check_image_quality(factory())
        print(f"{name:>8}: {result}")


   clear: {'is_usable': True, 'blur_score': 3829.0, 'brightness': 154.5, 'contrast': 124.4, 'reason': 'Image suitable for OCR'}
 blurred: {'is_usable': False, 'blur_score': 17.0, 'brightness': 154.5, 'contrast': 113.3, 'reason': 'Image is blurred'}
    dark: {'is_usable': False, 'blur_score': 234.1, 'brightness': 38.2, 'contrast': 30.7, 'reason': 'Image is underexposed'}


In [5]:
cases = {
    "clear": make_clear_image(),
    "blurred": make_blurred_image(),
    "dark": make_dark_image(),
}
for name, img in cases.items():
    print(name, "->", check_image_quality(img))


clear -> {'is_usable': True, 'blur_score': 3829.0, 'brightness': 154.5, 'contrast': 124.4, 'reason': 'Image suitable for OCR'}
blurred -> {'is_usable': False, 'blur_score': 17.0, 'brightness': 154.5, 'contrast': 113.3, 'reason': 'Image is blurred'}
dark -> {'is_usable': False, 'blur_score': 234.1, 'brightness': 38.2, 'contrast': 30.7, 'reason': 'Image is underexposed'}


## Question 3 — OCR confidence triage + safe corrections
HIGH ≥ 0.90 (may apply), REVIEW 0.60–0.89 (suggest only), LOW < 0.60 (never auto-correct). Original text is always preserved.

In [6]:
"""Question 3 — OCR review with confidence gating and safe corrections.

Validation strategy
-------------------
Every OCR field carries a confidence score and behaviour is gated on it:

* ``>= 0.90`` -> ``HIGH_CONFIDENCE``: a correction may be *applied*.
* ``0.60 .. 0.89`` -> ``REVIEW_REQUIRED``: a correction is only *suggested*
  together with a reason and is never applied automatically.
* ``< 0.60`` -> ``LOW_CONFIDENCE``: never auto-correct; flag for a human.

The original OCR text is always preserved (``CorrectionRecord.original``).
``clean_text`` only strips and collapses whitespace and never rewrites
characters; digit/letter fixes are emitted as suggestions only.

Medicine names are validated against a trusted dictionary: an exact
case-insensitive match first, then a ``difflib.get_close_matches`` fallback
with an explicit similarity cutoff.  Fuzzy hits are always marked
``needs_review``.
"""

from __future__ import annotations

import difflib
import re
from collections.abc import Iterable, Iterator, Sequence
from dataclasses import dataclass
from typing import Any

Q3_HIGH_THRESHOLD = 0.90
Q3_REVIEW_THRESHOLD = 0.60
Q3_FUZZY_CUTOFF = 0.80

# Common OCR digit -> letter confusions, only used to propose candidates.
_OCR_CONFUSIONS: dict[str, tuple[str, ...]] = {
    "0": ("o", "O"),
    "1": ("l", "I"),
    "2": ("Z",),
    "5": ("s", "S"),
    "6": ("G",),
    "8": ("B",),
}

TRUSTED_MEDICINES: set[str] = {
    "Amoxicillin",
    "Azithromycin",
    "Paracetamol",
    "Ibuprofen",
    "Metformin",
    "Atorvastatin",
    "Omeprazole",
    "Cetirizine",
    "Lisinopril",
    "Amlodipine",
}

OCR_FIXTURE: list[dict[str, Any]] = [
    {"id": "R1", "text": "Amoxicillin", "confidence": 0.95},
    {"id": "R2", "text": "Paracetamol", "confidence": 0.92},
    {"id": "R3", "text": "Ibuprofen", "confidence": 0.88},
    {"id": "R4", "text": "Metfromin", "confidence": 0.85},
    {"id": "R5", "text": "Am0xicillin", "confidence": 0.55},
    {"id": "R6", "text": "Cetirizin", "confidence": 0.72},
    {"id": "R7", "text": "Omeprazole", "confidence": 0.60},
    {"id": "R8", "text": "Aspirin", "confidence": 0.30},
]


@dataclass
class CorrectionRecord:
    """A single proposed correction; ``original`` is never overwritten."""

    original: str
    corrected: str
    reason: str
    confidence: float
    action: str  # "applied", "suggested" or "flagged"


def classify_confidence(confidence: float) -> str:
    """``>=0.90`` HIGH_CONFIDENCE, ``0.60..0.89`` REVIEW_REQUIRED, else LOW."""
    if confidence >= Q3_HIGH_THRESHOLD:
        return "HIGH_CONFIDENCE"
    if confidence >= Q3_REVIEW_THRESHOLD:
        return "REVIEW_REQUIRED"
    return "LOW_CONFIDENCE"


def clean_text(text: str) -> str:
    """Strip and collapse whitespace without changing any characters."""
    return re.sub(r"\s+", " ", text or "").strip()


def _confusion_candidates(name: str) -> Iterator[str]:
    """Yield single-character OCR repairs of ``name`` (suggestions only)."""
    for index, char in enumerate(name):
        for replacement in _OCR_CONFUSIONS.get(char, ()):
            yield name[:index] + replacement + name[index + 1 :]


def validate_medicine(
    name: str, trusted_dict: Iterable[str] | None = None
) -> dict[str, Any]:
    """Validate a medicine name against a trusted dictionary.

    Returns ``{valid, matched, method, needs_review}``.  Exact matches are
    ``method="exact"``; fuzzy hits are ``method="fuzzy"`` and always require
    review; anything else is ``method="none"``.
    """
    trusted = set(TRUSTED_MEDICINES if trusted_dict is None else trusted_dict)
    lookup = {med.lower(): med for med in trusted}
    cleaned = clean_text(name)

    if not cleaned:
        return {"valid": False, "matched": None, "method": "none", "needs_review": True}

    exact = lookup.get(cleaned.lower())
    if exact is not None:
        return {"valid": True, "matched": exact, "method": "exact", "needs_review": False}

    close = difflib.get_close_matches(cleaned.lower(), list(lookup), n=1, cutoff=Q3_FUZZY_CUTOFF)
    if close:
        return {
            "valid": True,
            "matched": lookup[close[0]],
            "method": "fuzzy",
            "needs_review": True,
        }
    return {"valid": False, "matched": None, "method": "none", "needs_review": True}


def _build_correction(
    original: str,
    cleaned: str,
    confidence: float,
    category: str,
    trusted: set[str],
) -> CorrectionRecord | None:
    """Decide a safe correction for one field, honouring the confidence gate."""
    result = validate_medicine(cleaned, trusted)
    if result["valid"] and result["method"] == "exact":
        return None  # already trustworthy, nothing to correct

    for candidate in _confusion_candidates(cleaned):
        repaired = validate_medicine(candidate, trusted)
        if repaired["valid"] and repaired["method"] == "exact":
            if category == "HIGH_CONFIDENCE":
                return CorrectionRecord(
                    original, candidate, "high-confidence OCR character correction",
                    confidence, "applied",
                )
            reason = (
                "medium confidence: character correction suggested for review"
                if category == "REVIEW_REQUIRED"
                else "low confidence: human review required before correcting"
            )
            return CorrectionRecord(original, candidate, reason, confidence, "suggested")

    if result["valid"] and result["method"] == "fuzzy":
        reason = (
            "fuzzy match to trusted medicine; confirm before applying"
            if category != "LOW_CONFIDENCE"
            else "low confidence: human review required before correcting"
        )
        return CorrectionRecord(original, result["matched"], reason, confidence, "suggested")

    if category == "LOW_CONFIDENCE":
        return CorrectionRecord(
            original, cleaned, "low confidence: no safe correction, flagged for human",
            confidence, "flagged",
        )
    return None


def review_ocr_results(
    results: Sequence[dict[str, Any]],
    medicine_dict: Iterable[str] | None = None,
    corrections: list[CorrectionRecord] | None = None,
) -> list[dict[str, Any]]:
    """Review OCR results, returning ``{original, cleaned, category, correction}``.

    ``corrections`` is an optional store; every emitted ``CorrectionRecord`` is
    appended to it so the original text stays auditable.  ``correction`` in the
    output is ``None`` or ``{corrected_text, reason, applied}``.
    """
    trusted = set(TRUSTED_MEDICINES if medicine_dict is None else medicine_dict)
    output: list[dict[str, Any]] = []

    for item in results:
        original = item.get("text", "")
        confidence = item.get("confidence", 0.0)
        cleaned = clean_text(original)
        category = classify_confidence(confidence)

        correction = _build_correction(original, cleaned, confidence, category, trusted)
        if correction is not None and corrections is not None:
            corrections.append(correction)

        correction_out = (
            {
                "corrected_text": correction.corrected,
                "reason": correction.reason,
                "applied": correction.action == "applied",
            }
            if correction is not None
            else None
        )
        output.append(
            {
                "original": original,
                "cleaned": cleaned,
                "category": category,
                "correction": correction_out,
            }
        )
    return output


if __name__ == "__main__":  # pragma: no cover - demo output
    store: list[CorrectionRecord] = []
    for row in review_ocr_results(OCR_FIXTURE, corrections=store):
        suggestion = row["correction"] or {}
        corrected = suggestion.get("corrected_text")
        applied = suggestion.get("applied")
        print(
            f"{row['original']!r} -> {row['category']} "
            f"(cleaned={row['cleaned']!r}, corrected={corrected}, applied={applied})"
        )
    print("Corrections recorded:", len(store))


'Amoxicillin' -> HIGH_CONFIDENCE (cleaned='Amoxicillin', corrected=None, applied=None)
'Paracetamol' -> HIGH_CONFIDENCE (cleaned='Paracetamol', corrected=None, applied=None)
'Ibuprofen' -> REVIEW_REQUIRED (cleaned='Ibuprofen', corrected=None, applied=None)
'Metfromin' -> REVIEW_REQUIRED (cleaned='Metfromin', corrected=Metformin, applied=False)
'Am0xicillin' -> LOW_CONFIDENCE (cleaned='Am0xicillin', corrected=Amoxicillin, applied=False)
'Cetirizin' -> REVIEW_REQUIRED (cleaned='Cetirizin', corrected=Cetirizine, applied=False)
'Omeprazole' -> REVIEW_REQUIRED (cleaned='Omeprazole', corrected=None, applied=None)
'Aspirin' -> LOW_CONFIDENCE (cleaned='Aspirin', corrected=Aspirin, applied=False)
Corrections recorded: 4


In [7]:
store = []
for r in review_ocr_results(OCR_FIXTURE, corrections=store):
    print(r["original"], "|", r["category"], "|", r["correction"])
print("\ncorrection store preserves original:", store[0].original, "->", store[0].corrected)
print(validate_medicine("Metfromin"))
print(validate_medicine("Amoxicillin"))


Amoxicillin | HIGH_CONFIDENCE | None
Paracetamol | HIGH_CONFIDENCE | None
Ibuprofen | REVIEW_REQUIRED | None
Metfromin | REVIEW_REQUIRED | {'corrected_text': 'Metformin', 'reason': 'fuzzy match to trusted medicine; confirm before applying', 'applied': False}
Am0xicillin | LOW_CONFIDENCE | {'corrected_text': 'Amoxicillin', 'reason': 'low confidence: human review required before correcting', 'applied': False}
Cetirizin | REVIEW_REQUIRED | {'corrected_text': 'Cetirizine', 'reason': 'fuzzy match to trusted medicine; confirm before applying', 'applied': False}
Omeprazole | REVIEW_REQUIRED | None
Aspirin | LOW_CONFIDENCE | {'corrected_text': 'Aspirin', 'reason': 'low confidence: no safe correction, flagged for human', 'applied': False}

correction store preserves original: Metfromin -> Metformin
{'valid': True, 'matched': 'Metformin', 'method': 'fuzzy', 'needs_review': True}
{'valid': True, 'matched': 'Amoxicillin', 'method': 'exact', 'needs_review': False}


## Question 4 — Handwritten prescription text → validated JSON
Never guess: missing medicine → INVALID, bad duration / swapped strength-frequency → REVIEW_REQUIRED with explicit `issues`.

In [8]:
"""Question 4 — structured prescription extraction with schema validation.

LLM-extraction-with-schema-validation pattern
---------------------------------------------
1. Prompt the LLM to return a JSON object matching a fixed schema
   (``patient_name``, ``medicine_name``, ``strength``, ``frequency``,
   ``duration``).
2. ``json.loads`` the reply into a dict.
3. Run the *same* validator used for any other source on that dict.
4. Reject/repair loop: if the status is ``INVALID`` or ``REVIEW_REQUIRED`` the
   issues list is fed back to the model (or a human) for correction before the
   record is accepted.

The validator never guesses: missing or ambiguous fields produce an explicit
``issues`` entry and a ``REVIEW_REQUIRED`` status, and a missing medicine name
is ``INVALID`` outright.
"""

from __future__ import annotations

import re
from typing import Any

REQUIRED_FIELDS = ("patient_name", "medicine_name", "strength", "frequency", "duration")

_DURATION_RE = re.compile(r"^\d+\s*(day|days|week|weeks|month|months)$", re.IGNORECASE)
_DOSE_PATTERN = re.compile(
    r"\d+(?:\.\d+)?\s*(?:mg/ml|mg|g|mcg|ml|tablets?|capsules?)", re.IGNORECASE
)
_FREQUENCY_KEYWORDS = (
    "daily",
    "once",
    "twice",
    "thrice",
    "od",
    "bd",
    "tds",
    "qid",
    "hour",
    "hours",
    "morning",
    "night",
    "weekly",
    "every",
)

_FIELD_ALIASES = {
    "patient": "patient_name",
    "patient name": "patient_name",
    "medicine": "medicine_name",
    "medicine name": "medicine_name",
    "drug": "medicine_name",
    "strength": "strength",
    "dose": "strength",
    "frequency": "frequency",
    "duration": "duration",
}


def parse_prescription_text(text: str) -> dict[str, str]:
    """Parse ``Key: value`` lines into a canonical prescription dict.

    Keys are matched case-insensitively; ``patient``/``medicine`` map to
    ``patient_name``/``medicine_name``.  Unrecognised lines are ignored.
    """
    record: dict[str, str] = {}
    for raw_line in (text or "").splitlines():
        line = raw_line.strip()
        if not line or ":" not in line:
            continue
        key, _, value = line.partition(":")
        canonical = _FIELD_ALIASES.get(key.strip().lower())
        if canonical is not None:
            record[canonical] = value.strip()
    return record


def validate_prescription(record: dict[str, Any]) -> dict[str, Any]:
    """Validate a parsed prescription without guessing.

    Returns ``{"status": ..., "issues": [...]}`` where status is ``VALID``
    (zero issues), ``INVALID`` (medicine name missing) or ``REVIEW_REQUIRED``.
    """
    issues: list[str] = []

    for field in REQUIRED_FIELDS:
        if not str(record.get(field) or "").strip():
            issues.append(f"missing required field: {field}")

    medicine = str(record.get("medicine_name") or "").strip()
    strength = str(record.get("strength") or "").strip()
    frequency = str(record.get("frequency") or "").strip()
    duration = str(record.get("duration") or "").strip()

    # Swap detection: strength holding a frequency keyword, or frequency
    # holding a dose pattern, means the two fields were crossed.
    if strength and frequency:
        strength_is_frequency = any(k in strength.lower() for k in _FREQUENCY_KEYWORDS)
        frequency_is_dose = _DOSE_PATTERN.search(frequency) is not None
        if strength_is_frequency or frequency_is_dose:
            issues.append("strength and frequency appear swapped")

    if duration and not _DURATION_RE.match(duration):
        issues.append(f"invalid or unreadable duration: {duration!r}")

    if strength and not _DOSE_PATTERN.search(strength):
        if not any(k in strength.lower() for k in _FREQUENCY_KEYWORDS):
            issues.append(f"strength does not contain a recognised dose: {strength!r}")

    if not medicine:
        status = "INVALID"
    elif issues:
        status = "REVIEW_REQUIRED"
    else:
        status = "VALID"
    return {"status": status, "issues": issues}


def build_prescription_json(text: str) -> dict[str, Any]:
    """Parse ``text`` and produce a schema-validated prescription record."""
    record = parse_prescription_text(text)
    result = validate_prescription(record)
    return {
        "patient_name": record.get("patient_name", ""),
        "medicine_name": record.get("medicine_name", ""),
        "strength": record.get("strength", ""),
        "frequency": record.get("frequency", ""),
        "duration": record.get("duration", ""),
        "validation_status": result["status"],
        "issues": result["issues"],
    }


if __name__ == "__main__":  # pragma: no cover - demo output
    valid_text = (
        "Patient: John Doe\n"
        "Medicine: Amoxicillin\n"
        "Strength: 500 mg\n"
        "Frequency: twice daily\n"
        "Duration: 7 days\n"
    )
    print("VALID example:", build_prescription_json(valid_text))

    swapped_text = (
        "Patient: Jane Roe\n"
        "Medicine: Ibuprofen\n"
        "Strength: twice daily\n"
        "Frequency: 400 mg\n"
        "Duration: 5 days\n"
    )
    print("SWAPPED example:", build_prescription_json(swapped_text))

    incomplete_text = "Patient: Sam Poe\nStrength: 10 mg\nFrequency: once daily\n"
    print("INCOMPLETE example:", build_prescription_json(incomplete_text))

VALID example: {'patient_name': 'John Doe', 'medicine_name': 'Amoxicillin', 'strength': '500 mg', 'frequency': 'twice daily', 'duration': '7 days', 'validation_status': 'VALID', 'issues': []}
SWAPPED example: {'patient_name': 'Jane Roe', 'medicine_name': 'Ibuprofen', 'strength': 'twice daily', 'frequency': '400 mg', 'duration': '5 days', 'validation_status': 'REVIEW_REQUIRED', 'issues': ['strength and frequency appear swapped']}
INCOMPLETE example: {'patient_name': 'Sam Poe', 'medicine_name': '', 'strength': '10 mg', 'frequency': 'once daily', 'duration': '', 'validation_status': 'INVALID', 'issues': ['missing required field: medicine_name', 'missing required field: duration']}


In [9]:
import json
valid = "Patient: Rahul Sharma\nMedicine: Paracetamol\nStrength: 500 mg\nFrequency: Twice daily\nDuration: 5 days"
missing_med = "Patient: Rahul Sharma\nStrength: 500 mg\nFrequency: Twice daily\nDuration: 5 days"
swapped = "Patient: Rahul Sharma\nMedicine: Paracetamol\nStrength: Twice daily\nFrequency: 500 mg\nDuration: 5 days"
bad_dur = "Patient: Rahul Sharma\nMedicine: Paracetamol\nStrength: 500 mg\nFrequency: Twice daily\nDuration: sometime"
for label, text in [("VALID", valid), ("MISSING_MEDICINE", missing_med),
                    ("SWAPPED", swapped), ("BAD_DURATION", bad_dur)]:
    print("==", label)
    print(json.dumps(build_prescription_json(text), indent=2))


== VALID
{
  "patient_name": "Rahul Sharma",
  "medicine_name": "Paracetamol",
  "strength": "500 mg",
  "frequency": "Twice daily",
  "duration": "5 days",
  "validation_status": "VALID",
  "issues": []
}
== MISSING_MEDICINE
{
  "patient_name": "Rahul Sharma",
  "medicine_name": "",
  "strength": "500 mg",
  "frequency": "Twice daily",
  "duration": "5 days",
  "validation_status": "INVALID",
  "issues": [
    "missing required field: medicine_name"
  ]
}
== SWAPPED
{
  "patient_name": "Rahul Sharma",
  "medicine_name": "Paracetamol",
  "strength": "Twice daily",
  "frequency": "500 mg",
  "duration": "5 days",
  "validation_status": "REVIEW_REQUIRED",
  "issues": [
    "strength and frequency appear swapped"
  ]
}
== BAD_DURATION
{
  "patient_name": "Rahul Sharma",
  "medicine_name": "Paracetamol",
  "strength": "500 mg",
  "frequency": "Twice daily",
  "duration": "sometime",
  "validation_status": "REVIEW_REQUIRED",
  "issues": [
    "invalid or unreadable duration: 'sometime'"
  ]

## Question 5 — Complete image → structured pipeline
Stages: preprocess → quality gate → intelligent routing (junk short-circuits with $0 LLM cost) → simulated OCR extraction → regex validation → cache → JSON.

In [10]:
"""Question 5 - image -> structured prescription pipeline (offline, $0 API).

Accepts a camera image of a prescription, preprocesses it, gates quality, routes
intelligently, extracts text with a deterministic simulated OCR engine, validates
fields with regexes, caches, and returns structured JSON plus latency/cost metrics.

Design decisions: the local-first quality gate (inline, dependency-free) rejects
junk before any LLM, saving 100% of its cost; results are cached by image hash so
repeats cost $0; deterministic regex validation + confidence triage run before any
LLM; Q2/Q3/Q4 helpers are imported lazily inside try/except with identical local
fallbacks so this file runs even if they are absent; seeded images only.

TIMING / COST (simulated, offline, single image)
Stage                  Latency   LLM cost  Notes
1 preprocess           ~200ms    $0.000    cv2 fastNlMeansDenoising
2 quality_gate         ~3ms      $0.000    Laplacian variance + brightness
3 intelligent_routing  <0.1ms    $0.000    junk -> early exit, $0
4 text_extraction      <1ms      $0.002    SimulatedOCREngine (offline)
5 validation           <1ms      $0.000    regex + confidence triage
6 optimize (cache)     <0.1ms    $0.000    hit -> $0, ~0 ms
7 structured_output    <0.1ms    $0.000    JSON assembly
TOTAL fresh good       ~220ms    $0.002    1 simulated LLM call
TOTAL junk image       ~225ms    $0.000    short-circuit at stage 3
TOTAL cache hit        ~0.6ms    $0.000    0 LLM calls
Run: execute this notebook top-to-bottom (Q5 demo runs under __main__).
"""

from __future__ import annotations

import hashlib
import importlib
import json
import re
import time
from copy import deepcopy
from typing import Any

import cv2
import numpy as np

COST_PER_LLM_CALL_USD = 0.002  # documented SIMULATION, not a real price
BLUR_MIN = 100.0               # Laplacian variance below -> blurry
BRIGHT_MIN, BRIGHT_MAX = 80.0, 250.0
CONTRAST_MIN = 55.0            # std below -> low contrast -> enhance
MAX_SIDE = 1000                # resize long edge to at most this

DOSE_RE = re.compile(r"\b\d+(?:\.\d+)?\s*(?:mg|mcg|µg|g|ml|iu|%)\b", re.I)
DURATION_RE = re.compile(r"\b\d+\s*(?:day|days|week|weeks|month|months)\b", re.I)
FREQ_RE = re.compile(
    r"\b(?:once|twice|thrice|daily|bd|tds|qid|od|bid|tid|every\s+\d+\s*hours?|"
    r"\d\s*-\s*\d\s*-\s*\d)\b",
    re.I,
)
FIELD_PATTERNS = {
    "patient_name": re.compile(r"patient\s*[:\-]\s*(.+)", re.I),
    "medicine_name": re.compile(r"(?:medicine|drug)\s*[:\-]\s*(.+)", re.I),
    "strength": re.compile(r"strength\s*[:\-]\s*(.+)", re.I),
    "frequency": re.compile(r"frequency\s*[:\-]\s*(.+)", re.I),
    "duration": re.compile(r"duration\s*[:\-]\s*(.+)", re.I),
}
REQUIRED_FIELDS = ("patient_name", "medicine_name", "strength", "frequency", "duration")

def _optional(module: str, attr: str) -> Any:
    """Lazily import an optional Q2/Q3/Q4 helper; None if absent (fallback)."""
    for name in (module,):
        try:
            return getattr(importlib.import_module(name), attr)
        except Exception:
            continue
    return None

# Optional reuse of earlier questions - never a hard top-level dependency.
_q2_quality_gate = _optional("q2_image_quality", "check_image_quality")
_q3_triage = _optional("q3_ocr", "classify_confidence")
_q4_parse_fields = _optional("q4_structured", "parse_prescription_text")

def _local_triage(confidence: float) -> str:
    """Q3 fallback: HIGH >=0.90, REVIEW >=0.60, else LOW (one-liner)."""
    return "HIGH_CONFIDENCE" if confidence >= 0.90 else ("REVIEW_REQUIRED" if confidence >= 0.60 else "LOW_CONFIDENCE")

def _local_parse_fields(text: str) -> dict[str, str]:
    """Q4 fallback: pull labelled fields out of OCR text with regexes."""
    return {name: m.group(1).strip() for name, p in FIELD_PATTERNS.items()
            if (m := p.search(text))}

def _confidence_tier(confidence: float) -> str:
    """Map a confidence into HIGH / REVIEW / LOW (inline, no dependency)."""
    return "HIGH" if confidence >= 0.90 else ("REVIEW" if confidence >= 0.60 else "LOW")

class SimulatedOCREngine:
    """Deterministic offline stand-in for an OCR / LLM text extractor.
    Purpose/Input/Output: grayscale image (+ optional expected_text hint) ->
    ``[{"text", "confidence"}, ...]``. Impact: the only stage simulating an LLM
    call (1 call), so it dominates the $0.002 cost; deterministic and offline.
    """

    def __init__(self, base_confidence: float = 0.96) -> None:
        self.base_confidence: float = base_confidence

    @staticmethod
    def _ink_density(gray: np.ndarray) -> float:
        """Fraction of dark (ink) pixels, used to scale confidence."""
        return float(np.count_nonzero(gray < 128)) / float(gray.size)

    def _contour_heuristic(self, gray: np.ndarray) -> list[dict[str, Any]]:
        """Blank/contour fallback: cluster contours into text rows."""
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        ys = sorted(cv2.boundingRect(c)[1] for c in contours if cv2.contourArea(c) > 5)
        rows: list[int] = []
        for y in ys:
            if not rows or y - rows[-1] > 10:
                rows.append(y)
        conf = round(min(0.89, max(0.40, 0.45 + self._ink_density(gray) * 8.0)), 2)
        return [{"text": f"[unreadable_line_{i + 1}]", "confidence": conf}
                for i in range(max(1, len(rows)))]

    def extract(self, gray: np.ndarray, expected_text: str = "") -> list[dict[str, Any]]:
        """Use the hint when supplied, else the contour heuristic."""
        if expected_text.strip():
            conf = round(self.base_confidence if self._ink_density(gray) > 0.004 else 0.58, 2)
            return [{"text": ln.strip(), "confidence": conf}
                    for ln in expected_text.splitlines() if ln.strip()]
        return self._contour_heuristic(gray)

class TextExtractionPipeline:
    """Seven-stage image -> structured JSON pipeline with cost/latency metrics."""

    def __init__(self) -> None:
        self.ocr: SimulatedOCREngine = SimulatedOCREngine()
        self._cache: dict[str, tuple[dict[str, Any], dict[str, Any]]] = {}
        self._llm_calls_total: int = 0

    def preprocess(self, image: np.ndarray) -> np.ndarray:
        """Stage 1 - Image Preprocessing.
        Purpose/Input/Output: normalise a BGR/gray capture into a denoised, resized
        grayscale image. Impact: denoising lifts accuracy on noisy captures; this
        CPU work is the largest latency term but costs no LLM.
        """
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image.copy()
        h, w = gray.shape[:2]
        if max(h, w) > MAX_SIDE:
            s = MAX_SIDE / float(max(h, w))
            gray = cv2.resize(gray, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)
        try:
            gray = cv2.fastNlMeansDenoising(gray, None, 10, 7, 21)
        except Exception:  # pragma: no cover - defensive fallback
            gray = cv2.medianBlur(gray, 3)
        return gray

    def quality_gate(self, gray: np.ndarray) -> dict[str, Any]:
        """Stage 2 - Quality Gate (inline, dependency-free).
        Purpose/Input/Output: grayscale -> ``{is_usable, blur_score, brightness,
        contrast, reason}``. Impact: ~1 ms / $0; rejects junk up front so accuracy
        is protected and 100% downstream LLM cost is saved.
        """
        blur_score = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        brightness, contrast = float(gray.mean()), float(gray.std())
        problems: list[str] = []
        if blur_score < BLUR_MIN:
            problems.append("blurred")
        if not (BRIGHT_MIN <= brightness <= BRIGHT_MAX):
            problems.append("poorly illuminated")
        return {"is_usable": not problems, "blur_score": round(blur_score, 2),
                "brightness": round(brightness, 1), "contrast": round(contrast, 1),
                "reason": "ok" if not problems else "Image may be " + " or ".join(problems)}

    def intelligent_routing(self, quality: dict[str, Any]) -> dict[str, str]:
        """Stage 3 - Intelligent Routing.
        Purpose/Input/Output: quality dict -> ``{action, reason}`` where action is
        NEEDS_RETAKE / ENHANCE / TEXT_EXTRACTION. Impact: junk returns early ->
        0 LLM calls ($0); low contrast takes free local CLAHE.
        """
        if not quality["is_usable"]:
            return {"action": "NEEDS_RETAKE", "reason": quality["reason"]}
        if quality["contrast"] < CONTRAST_MIN:
            return {"action": "ENHANCE", "reason": "low contrast -> local enhance"}
        return {"action": "TEXT_EXTRACTION", "reason": "routed to local extractor"}

    def text_extraction(self, gray: np.ndarray, expected_text: str = "") -> list[dict[str, Any]]:
        """Stage 4 - AI-Based Text Extraction (simulated, offline).
        Purpose/Input/Output: image + hint -> ``{text, confidence}`` records.
        Impact: the only LLM-incurring stage; it increments the call counter so
        cost is attributed ($0.002 simulated) and is deterministic for accuracy.
        """
        self._llm_calls_total += 1
        return self.ocr.extract(gray, expected_text)

    def validation(
        self, lines: list[dict[str, Any]], quality: dict[str, Any]
    ) -> tuple[str, dict[str, str], dict[str, Any]]:
        """Stage 5 - Accuracy and Validation.
        Purpose/Input/Output: lines + quality -> ``(status, fields, report)`` with
        regex dose/duration checks and HIGH/REVIEW/LOW confidence triage. Impact:
        deterministic checks cost <1 ms / $0 and flag missing or ambiguous fields
        instead of guessing, so only truly uncertain cases reach human review.
        """
        text = "\n".join(ln["text"] for ln in lines)
        parser = _q4_parse_fields or _local_parse_fields
        try:
            fields = dict(parser(text))
        except Exception:  # pragma: no cover - defensive
            fields = _local_parse_fields(text)
        _ = _q3_triage or _local_triage  # Q3 hook kept for parity; triage is inline
        tiers = {ln["text"]: _confidence_tier(float(ln["confidence"])) for ln in lines}
        lowest = min((float(ln["confidence"]) for ln in lines), default=1.0)
        missing = [n for n in REQUIRED_FIELDS if not fields.get(n)]
        flags: list[str] = ["missing:" + ",".join(missing)] if missing else []
        if fields.get("strength") and not DOSE_RE.search(fields["strength"]):
            flags.append("strength_format")
        if fields.get("duration") and not DURATION_RE.search(fields["duration"]):
            flags.append("duration_format")
        if (fields.get("strength") and FREQ_RE.search(fields["strength"])
                and not DOSE_RE.search(fields["strength"])) or (
                fields.get("frequency") and DOSE_RE.search(fields["frequency"])
                and not FREQ_RE.search(fields["frequency"])):
            flags.append("strength_frequency_swapped")
        if not fields:
            status = "REVIEW_REQUIRED"
        elif missing:
            status = "INVALID" if "medicine_name" in missing else "REVIEW_REQUIRED"
        else:
            status = "VALID" if _confidence_tier(lowest) == "HIGH" and not flags else "REVIEW_REQUIRED"
        report = {"overall_tier": _confidence_tier(lowest),
                  "lowest_confidence": round(lowest, 2), "line_tiers": tiers,
                  "flags": flags, "usable": quality["is_usable"]}
        return status, fields, report

    def optimize(self, cache_key: str) -> tuple[dict[str, Any], dict[str, Any]] | None:
        """Stage 6 - Cost & Performance Optimization (cache lookup).
        Purpose/Input/Output: cache key -> cached ``(record, metrics)`` or None.
        Impact: a hit costs $0, makes 0 LLM calls, and is a ~0.1 ms dict lookup --
        the biggest end-to-end win for repeated images.
        """
        return self._cache.get(cache_key)

    @staticmethod
    def _cache_key(image: np.ndarray, expected_text: str) -> str:
        """Stable hash of image bytes (shape/dtype) salted with the hint."""
        d = hashlib.sha256()
        d.update(f"{image.shape}|{image.dtype}|{expected_text}".encode())
        d.update(np.ascontiguousarray(image).tobytes())
        return d.hexdigest()

    def structured_output(
        self, fields: dict[str, str], quality: dict[str, Any], confidence: dict[str, Any],
        status: str, lines: list[dict[str, Any]], metrics: dict[str, Any],
    ) -> dict[str, Any]:
        """Stage 7 - Structured Output.
        Purpose/Input/Output: validated parts -> JSON-serialisable prescription
        ``dict`` with an ``extraction_meta`` block. Impact: pure assembly, ~0 ms /
        $0; guarantees a stable contract whichever path ran.
        """
        record: dict[str, Any] = {n: fields.get(n, "") for n in REQUIRED_FIELDS}
        record["validation_status"] = status
        record["extraction_meta"] = {
            "latency_ms": metrics["latency_ms"], "llm_calls": metrics["llm_calls"],
            "cost_usd": metrics["cost_usd"], "quality": quality,
            "confidence": confidence, "text_lines": lines,
            "cached": metrics["cached"], "route": metrics["route"]}
        return record

    def run(
        self, image: np.ndarray, expected_text: str = ""
    ) -> tuple[dict[str, Any], dict[str, Any]]:
        """Run all seven stages; return ``(record, metrics)``.
        ``metrics`` = ``{latency_ms, llm_calls, cost_usd, cached, route, quality,
        stage_latency_ms}``. Latency uses ``time.perf_counter``; cost is the
        documented simulated ``llm_calls * 0.002`` USD.
        """
        started, stage_ms = time.perf_counter(), {}
        cache_key = self._cache_key(image, expected_text)

        def mark(stage: str, t0: float) -> None:
            stage_ms[stage] = round((time.perf_counter() - t0) * 1000.0, 3)

        cached = self.optimize(cache_key)  # Stage 6 read path
        if cached is not None:
            record = deepcopy(cached[0])
            latency_ms = round((time.perf_counter() - started) * 1000.0, 3)
            record["extraction_meta"].update(latency_ms=latency_ms, llm_calls=0,
                                             cost_usd=0.0, cached=True, route="CACHE_HIT")
            return record, {"latency_ms": latency_ms, "llm_calls": 0, "cost_usd": 0.0,
                            "cached": True, "route": "CACHE_HIT",
                            "quality": record["extraction_meta"].get("quality", {}),
                            "stage_latency_ms": {"optimize": latency_ms}}

        calls_before = self._llm_calls_total
        t0 = time.perf_counter(); gray = self.preprocess(image); mark("preprocess", t0)
        t0 = time.perf_counter(); quality = self.quality_gate(gray); mark("quality_gate", t0)
        t0 = time.perf_counter(); route = self.intelligent_routing(quality)
        mark("intelligent_routing", t0)

        lines: list[dict[str, Any]] = []
        fields: dict[str, str] = {}
        confidence: dict[str, Any] = {"overall_tier": "LOW", "lowest_confidence": 0.0,
                                      "line_tiers": {}, "flags": []}
        status = "NEEDS_RETAKE"
        if route["action"] != "NEEDS_RETAKE":
            if route["action"] == "ENHANCE":
                t0 = time.perf_counter()
                gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
                mark("enhance", t0)
            t0 = time.perf_counter(); lines = self.text_extraction(gray, expected_text)
            mark("text_extraction", t0)
            t0 = time.perf_counter(); status, fields, confidence = self.validation(lines, quality)
            mark("validation", t0)

        llm_calls = self._llm_calls_total - calls_before
        metrics: dict[str, Any] = {
            "latency_ms": round((time.perf_counter() - started) * 1000.0, 3),
            "llm_calls": llm_calls, "cost_usd": round(llm_calls * COST_PER_LLM_CALL_USD, 6),
            "cached": False, "route": route["action"], "quality": quality,
            "stage_latency_ms": stage_ms}
        t0 = time.perf_counter()
        record = self.structured_output(fields, quality, confidence, status, lines, metrics)
        mark("structured_output", t0)
        self._cache[cache_key] = (deepcopy(record), metrics)
        return record, metrics

_LINES = ["Patient: Rahul Sharma", "Medicine: Paracetamol", "Strength: 500 mg", "Frequency: Twice daily", "Duration: 5 days"]

def _clear_image() -> np.ndarray:
    """Synthetic white board with sharp printed lines (deterministic)."""
    img = np.full((240, 640, 3), 255, np.uint8)
    for i, line in enumerate(_LINES):
        cv2.putText(img, line, (20, 40 + i * 40), cv2.FONT_HERSHEY_SIMPLEX,
                    0.9, (0, 0, 0), 2, cv2.LINE_AA)
    return img

def _bad_image() -> np.ndarray:
    """Heavily blurred and underexposed capture (deterministic)."""
    blurred = cv2.GaussianBlur(_clear_image(), (41, 41), 0)
    return cv2.convertScaleAbs(blurred, alpha=0.3, beta=0)

if __name__ == "__main__":
    expected = "\n".join(_LINES)
    pipeline = TextExtractionPipeline()
    good, good_metrics = pipeline.run(_clear_image(), expected_text=expected)
    print("=== GOOD IMAGE ===")
    print(json.dumps(good, indent=2))
    print("metrics:", json.dumps(good_metrics, indent=2))
    bad, bad_metrics = pipeline.run(_bad_image())
    print("\n=== BAD IMAGE (junk) ===")
    print(json.dumps(bad, indent=2))
    print("metrics:", json.dumps(bad_metrics, indent=2))
    _, hit_metrics = pipeline.run(_clear_image(), expected_text=expected)
    print("\n=== CACHE HIT (repeat) ===")
    print("metrics:", json.dumps(hit_metrics, indent=2))


=== GOOD IMAGE ===
{
  "patient_name": "Rahul Sharma",
  "medicine_name": "Paracetamol",
  "strength": "500 mg",
  "frequency": "Twice daily",
  "duration": "5 days",
  "validation_status": "VALID",
  "extraction_meta": {
    "latency_ms": 124.501,
    "llm_calls": 1,
    "cost_usd": 0.002,
    "quality": {
      "is_usable": true,
      "blur_score": 3268.68,
      "brightness": 238.7,
      "contrast": 58.8,
      "reason": "ok"
    },
    "confidence": {
      "overall_tier": "HIGH",
      "lowest_confidence": 0.96,
      "line_tiers": {
        "Patient: Rahul Sharma": "HIGH",
        "Medicine: Paracetamol": "HIGH",
        "Strength: 500 mg": "HIGH",
        "Frequency: Twice daily": "HIGH",
        "Duration: 5 days": "HIGH"
      },
      "flags": [],
      "usable": true
    },
    "text_lines": [
      {
        "text": "Patient: Rahul Sharma",
        "confidence": 0.96
      },
      {
        "text": "Medicine: Paracetamol",
        "confidence": 0.96
      },
      {
    


=== BAD IMAGE (junk) ===
{
  "patient_name": "",
  "medicine_name": "",
  "strength": "",
  "frequency": "",
  "duration": "",
  "validation_status": "NEEDS_RETAKE",
  "extraction_meta": {
    "latency_ms": 103.915,
    "llm_calls": 0,
    "cost_usd": 0.0,
    "quality": {
      "is_usable": false,
      "blur_score": 0.51,
      "brightness": 71.4,
      "contrast": 8.1,
      "reason": "Image may be blurred or poorly illuminated"
    },
    "confidence": {
      "overall_tier": "LOW",
      "lowest_confidence": 0.0,
      "line_tiers": {},
      "flags": []
    },
    "text_lines": [],
    "cached": false,
    "route": "NEEDS_RETAKE"
  }
}
metrics: {
  "latency_ms": 103.915,
  "llm_calls": 0,
  "cost_usd": 0.0,
  "cached": false,
  "route": "NEEDS_RETAKE",
  "quality": {
    "is_usable": false,
    "blur_score": 0.51,
    "brightness": 71.4,
    "contrast": 8.1,
    "reason": "Image may be blurred or poorly illuminated"
  },
  "stage_latency_ms": {
    "preprocess": 102.401,
    "qu

In [11]:
import json
import cv2
import numpy as np
EXPECTED = ("Patient: Rahul Sharma\nMedicine: Paracetamol\nStrength: 500 mg\n"
            "Frequency: Twice daily\nDuration: 5 days")
def make_rx(seed=7):
    rng = np.random.default_rng(seed)
    img = np.full((240, 640, 3), 255, np.uint8)
    for i, line in enumerate(EXPECTED.splitlines()):
        cv2.putText(img, line, (20, 40 + i * 38), cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, (0, 0, 0), 2, cv2.LINE_AA)
    return np.clip(img.astype(np.int16) + rng.integers(-3, 4, img.shape), 0, 255).astype(np.uint8)
pipe = TextExtractionPipeline()
rec, metrics = pipe.run(make_rx(), expected_text=EXPECTED)
print(json.dumps(rec, indent=2))
print("metrics:", metrics)
bad = cv2.convertScaleAbs(cv2.GaussianBlur(make_rx(11), (41, 41), 0), alpha=0.3, beta=0)
rec2, m2 = pipe.run(bad)
print("bad image:", rec2["validation_status"], m2)
_, m3 = pipe.run(make_rx(), expected_text=EXPECTED)
print("cache hit:", m3)


{
  "patient_name": "Rahul Sharma",
  "medicine_name": "Paracetamol",
  "strength": "500 mg",
  "frequency": "Twice daily",
  "duration": "5 days",
  "validation_status": "VALID",
  "extraction_meta": {
    "latency_ms": 109.011,
    "llm_calls": 1,
    "cost_usd": 0.002,
    "quality": {
      "is_usable": true,
      "blur_score": 3146.02,
      "brightness": 240.5,
      "contrast": 53.7,
      "reason": "ok"
    },
    "confidence": {
      "overall_tier": "HIGH",
      "lowest_confidence": 0.96,
      "line_tiers": {
        "Patient: Rahul Sharma": "HIGH",
        "Medicine: Paracetamol": "HIGH",
        "Strength: 500 mg": "HIGH",
        "Frequency: Twice daily": "HIGH",
        "Duration: 5 days": "HIGH"
      },
      "flags": [],
      "usable": true
    },
    "text_lines": [
      {
        "text": "Patient: Rahul Sharma",
        "confidence": 0.96
      },
      {
        "text": "Medicine: Paracetamol",
        "confidence": 0.96
      },
      {
        "text": "Streng

bad image: NEEDS_RETAKE {'latency_ms': 106.581, 'llm_calls': 0, 'cost_usd': 0.0, 'cached': False, 'route': 'NEEDS_RETAKE', 'quality': {'is_usable': False, 'blur_score': 0.45, 'brightness': 72.1, 'contrast': 7.3, 'reason': 'Image may be blurred or poorly illuminated'}, 'stage_latency_ms': {'preprocess': 105.065, 'quality_gate': 1.024, 'intelligent_routing': 0.001, 'structured_output': 0.004}}
cache hit: {'latency_ms': 0.382, 'llm_calls': 0, 'cost_usd': 0.0, 'cached': True, 'route': 'CACHE_HIT', 'quality': {'is_usable': True, 'blur_score': 3146.02, 'brightness': 240.5, 'contrast': 53.7, 'reason': 'ok'}, 'stage_latency_ms': {'optimize': 0.382}}


## Question 6 — Multi-knowledge-base chatbot
Per-project retrieval isolation (no cross-document leakage), ACL enforcement, source citations, history, topic switching, NO_EVIDENCE / CONTRADICTION / DENIED guardrails, audit log.

In [12]:
"""Question 6 -- Multi-knowledge-base chatbot prototype.

Components
----------
* KB store   ``KnowledgeBase(project_id, docs)``; a doc is ``{"doc_id", "text"}``.
* ACL        ``AccessControl.grants`` maps ``user_id -> {project_ids}`` and is
             checked on every project selection and on implicit topic switch.
* Retriever  Token-overlap scorer confined to exactly ONE project:
             ``score = |query n doc| / sqrt(|doc|)``. Because it only ever sees
             the active project's docs it is anti-leak by construction.
* Dialogue   ``Chatbot``: login, project selection, per-(user, project) history,
             thin follow-up merging, explicit (``project <id>``) / implicit
             (project-name keyword) topic switching.
* Guardrail  ``ANSWERED`` / ``NO_EVIDENCE`` / ``CONTRADICTION`` / ``DENIED`` plus
             an audit log of ``(user, project, question, sources)``.

Seed: alpha (20 annual leaves), beta (``deploy.sh --prod``), gamma (30-day
refund) with an intentional contradiction in gamma (30-day physical vs 7-day
digital).

Limitations / production improvements
-------------------------------------
* Lexical retrieval misses paraphrases -> embeddings + a vector DB
  (pgvector/FAISS/Qdrant) with a cross-encoder re-ranker.
* Template answers -> ground an LLM on the retrieved evidence only.
* Keyword contradiction heuristic -> NLI / entailment models.
* No PII redaction or per-document ACLs -> add PII scrubbing, doc-level perms.
* In-memory state, no rate limiting -> durable audit store + quotas.
"""

from __future__ import annotations

import math
import re
from dataclasses import dataclass

ANSWERED = "ANSWERED"
NO_EVIDENCE = "NO_EVIDENCE"
CONTRADICTION = "CONTRADICTION"
DENIED = "DENIED"

_TOKEN_RE = re.compile(r"[a-z0-9]+")
NEGATION_MARKERS = {"not", "never", "except", "no", "cannot", "without", "none"}

# Stripped from scoring only (never from NEGATION_MARKERS) so filler words such
# as "is"/"of"/"the" cannot manufacture spurious matches.
STOPWORDS = {
    "a", "an", "the", "is", "are", "was", "were", "be", "been", "of", "to",
    "for", "and", "or", "in", "on", "at", "by", "with", "do", "does", "did",
    "i", "you", "we", "me", "my", "our", "us", "it", "this", "that", "these",
    "those", "what", "which", "who", "how", "many", "much", "about", "show",
    "tell", "give", "get", "gives", "can", "could", "would", "should", "please",
    "now", "s", "t",
}


def tokenize(text: str) -> list[str]:
    """Lowercase and split into alphanumeric tokens."""
    return _TOKEN_RE.findall(text.lower())


def content_tokens(text: str) -> list[str]:
    """Tokens without stopwords; used for scoring and overlap counting."""
    return [token for token in tokenize(text) if token not in STOPWORDS]


@dataclass
class KnowledgeBase:
    """One isolated project corpus."""

    project_id: str
    docs: list[dict[str, str]]


@dataclass
class AccessControl:
    """User -> granted project ids."""

    grants: dict[str, set[str]]

    def can_access(self, user_id: str, project_id: str) -> bool:
        return project_id in self.grants.get(user_id, set())


class Retriever:
    """Token-overlap retriever scoped to a single project per call."""

    def __init__(
        self,
        kbs: dict[str, KnowledgeBase],
        threshold: float = 0.35,
        high_threshold: float = 0.30,
    ) -> None:
        self._index: dict[str, list[tuple[dict[str, str], list[str]]]] = {
            pid: [(doc, content_tokens(doc["text"])) for doc in kb.docs]
            for pid, kb in kbs.items()
        }
        self.threshold: float = threshold
        self.high_threshold: float = high_threshold

    def retrieve(
        self, project_id: str, query: str, top_k: int
    ) -> list[tuple[dict[str, str], float]]:
        """Return up to ``top_k`` ``(doc, score)`` from ``project_id`` only."""
        q = set(content_tokens(query))
        scored: list[tuple[dict[str, str], float]] = []
        for doc, dtokens in self._index.get(project_id, []):
            overlap = q & set(dtokens)
            if not overlap:
                continue
            score = len(overlap) / math.sqrt(len(dtokens)) if dtokens else 0.0
            scored.append((doc, score))
        scored.sort(key=lambda pair: (-pair[1], pair[0]["doc_id"]))
        return scored[:top_k]

    def overlap_count(self, project_id: str, query: str) -> int:
        union: set[str] = set()
        for _, dtokens in self._index.get(project_id, []):
            union |= set(dtokens)
        return len(set(content_tokens(query)) & union)

    def find_contradiction(
        self, project_id: str, scored: list[tuple[dict[str, str], float]]
    ) -> tuple[dict[str, str], dict[str, str]] | None:
        """Return two strong opposing docs (one negated, one not)."""
        tokens = {d["doc_id"]: set(t) for d, t in self._index.get(project_id, [])}
        strong = [(d, s) for d, s in scored if s >= self.high_threshold]
        for i in range(len(strong)):
            for j in range(i + 1, len(strong)):
                d1, d2 = strong[i][0], strong[j][0]
                t1, t2 = tokens[d1["doc_id"]], tokens[d2["doc_id"]]
                if bool(t1 & NEGATION_MARKERS) != bool(t2 & NEGATION_MARKERS):
                    if len((t1 & t2) - NEGATION_MARKERS) >= 3:
                        return d1, d2
        return None


class Chatbot:
    """Multi-project chatbot with ACL, citations, history and guardrails."""

    def __init__(
        self,
        kbs: dict[str, KnowledgeBase],
        acl: AccessControl,
        retriever: Retriever | None = None,
    ) -> None:
        self.kbs: dict[str, KnowledgeBase] = kbs
        self.acl: AccessControl = acl
        self.retriever: Retriever = retriever or Retriever(kbs)
        self._user: str | None = None
        self._project: str | None = None
        self.history: dict[tuple[str, str], list[str]] = {}
        self.audit_log: list[tuple[str, str, str, list[str]]] = []

    def login(self, user_id: str) -> None:
        self._user = user_id
        self._project = None

    def select_project(self, project_id: str) -> None:
        if self._user is None:
            raise RuntimeError("login required before selecting a project")
        if project_id not in self.kbs:
            raise KeyError(f"unknown project '{project_id}'")
        if not self.acl.can_access(self._user, project_id):
            raise PermissionError(
                f"user '{self._user}' is not authorized for project '{project_id}'"
            )
        self._project = project_id

    @property
    def active_project(self) -> str | None:
        return self._project

    def ask(self, question: str, top_k: int = 2) -> dict[str, object]:
        if self._user is None:
            return self._response(DENIED, "Please log in first.", [], None)

        target = self._detect_project(question)
        if target is not None and target != self._project:
            if not self.acl.can_access(self._user, target):
                self._audit(question, [])
                return self._response(
                    DENIED,
                    f"Access denied: you are not authorized for project '{target}'.",
                    [],
                    self._project,
                )
            self._project = target

        if self._project is None:
            return self._response(DENIED, "Select a project first.", [], None)

        project = self._project
        history = self.history.setdefault((self._user, project), [])
        query = question
        if history and self.retriever.overlap_count(project, question) < 3:
            query = f"{history[-1]} {question}"

        scored = self.retriever.retrieve(project, query, top_k)
        history.append(question)

        if not scored or scored[0][1] < self.retriever.threshold:
            self._audit(question, [])
            return self._response(
                NO_EVIDENCE,
                f"I don't have evidence in {project} to answer that.",
                [],
                project,
            )

        pair = self.retriever.find_contradiction(project, scored)
        if pair is not None:
            sources = [pair[0]["doc_id"], pair[1]["doc_id"]]
            self._audit(question, sources)
            return self._response(
                CONTRADICTION,
                f"Conflicting evidence in {project} "
                f"({sources[0]} vs {sources[1]}). Could you clarify which applies?",
                sources,
                project,
            )

        best = scored[0][0]
        sources = [doc["doc_id"] for doc, s in scored if s >= self.retriever.threshold]
        self._audit(question, sources)
        return self._response(
            ANSWERED, f"[{best['doc_id']}] {best['text']}", sources, project
        )

    def _detect_project(self, question: str) -> str | None:
        tokens = tokenize(question)
        for i, token in enumerate(tokens):
            if token == "project" and i + 1 < len(tokens) and tokens[i + 1] in self.kbs:
                return tokens[i + 1]
        for token in tokens:
            if token in self.kbs:
                return token
        return None

    def _audit(self, question: str, sources: list[str]) -> None:
        self.audit_log.append(
            (self._user or "-", self._project or "-", question, list(sources))
        )

    @staticmethod
    def _response(
        mode: str, answer: str, sources: list[str], project_id: str | None
    ) -> dict[str, object]:
        return {
            "answer": answer,
            "sources": sources,
            "project_id": project_id,
            "mode": mode,
        }


def build_seed_environment() -> tuple[dict[str, KnowledgeBase], AccessControl]:
    """Create the three demo projects and the ACL."""

    def kb(pid: str, docs: list[tuple[str, str]]) -> KnowledgeBase:
        return KnowledgeBase(pid, [{"doc_id": i, "text": t} for i, t in docs])

    kbs = {
        "alpha": kb("alpha", [
            ("a1", "Alpha leave policy: employees receive 20 annual leaves per year. Annual leave accrues monthly and rolls over up to 5 days."),
            ("a2", "Alpha sick leave policy: employees receive 10 sick leaves per year and sick leave requires a medical certificate."),
            ("a3", "Alpha parental leave: 12 weeks of paid parental leave is available for primary caregivers."),
        ]),
        "beta": kb("beta", [
            ("b1", "Beta service deployment guide: run deploy.sh --prod to release to production. Always run the test suite first."),
            ("b2", "Beta rollback guide: if a deployment fails, run deploy.sh --rollback to restore the previous version."),
            ("b3", "Beta staging guide: deploy.sh --staging deploys the service to the staging environment for verification."),
        ]),
        "gamma": kb("gamma", [
            ("g1", "Gamma refund policy: customers have a 30-day refund window for physical goods. Refunds are processed within 5 days."),
            ("g2", "Gamma digital goods refund policy: digital products have a 7-day refund window and are not eligible for refund after 7 days."),
            ("g3", "Gamma shipping policy: standard shipping takes 5-7 business days and express shipping takes 2 days."),
        ]),
    }
    acl = AccessControl({"alice": {"alpha", "beta", "gamma"}, "bob": {"beta"}})
    return kbs, acl


def demo_conversations() -> list[dict[str, object]]:
    """Return five failure-case walkthroughs as data."""
    specs: list[tuple[str, str, list[str], str, str]] = [
        (
            "alice", "beta", ["how many annual leaves do employees get?"],
            "Cross-project leak attempt (alpha question while in beta)",
            "The retriever only indexes the active project, so no alpha doc_id can "
            "appear while beta is selected.",
        ),
        (
            "bob", "beta", ["project alpha what is the annual leave policy?"],
            "Unauthorized project access (bob asks alpha)",
            "ACL is checked on every selection and implicit switch; unauthorized "
            "targets return DENIED with no sources.",
        ),
        (
            "alice", "alpha",
            ["how many annual leaves do employees get?",
             "now show me the gamma refund window"],
            "Topic switch mid-conversation (alpha -> gamma)",
            "A project keyword, or an explicit 'project <id>' phrase, switches the "
            "active corpus (ACL permitting).",
        ),
        (
            "alice", "alpha", ["what is the capital of France?"],
            "Missing evidence (out-of-scope question)",
            "When the best score is below threshold the bot refuses with "
            "NO_EVIDENCE instead of hallucinating.",
        ),
        (
            "alice", "gamma", ["what is the refund window in days?"],
            "Contradictory documents (30-day vs 7-day refund)",
            "Two high-scoring docs with opposing negation and shared keywords "
            "raise CONTRADICTION and cite both for clarification.",
        ),
    ]
    demos: list[dict[str, object]] = []
    for user, project, questions, title, fix in specs:
        kbs, acl = build_seed_environment()
        bot = Chatbot(kbs, acl)
        bot.login(user)
        bot.select_project(project)
        turns: list[dict[str, object]] = []
        for question in questions:
            resp = bot.ask(question)
            turns.append(
                {
                    "question": question,
                    "mode": resp["mode"],
                    "sources": resp["sources"],
                    "answer": resp["answer"],
                }
            )
        demos.append({"title": title, "turns": turns, "fix_explanation": fix})
    return demos


if __name__ == "__main__":
    kbs, acl = build_seed_environment()
    bot = Chatbot(kbs, acl)
    bot.login("alice")

    print("== Sample Q&A 1 ==")
    bot.select_project("alpha")
    print(bot.ask("how many annual leaves do employees get?"))

    print("== Sample Q&A 2 ==")
    bot.select_project("beta")
    print(bot.ask("how do I deploy to production?"))

    print("== Failure demo: cross-project leak attempt ==")
    print(bot.ask("how many annual leaves do employees get?"))


== Sample Q&A 1 ==
{'answer': '[a1] Alpha leave policy: employees receive 20 annual leaves per year. Annual leave accrues monthly and rolls over up to 5 days.', 'sources': ['a1', 'a2'], 'project_id': 'alpha', 'mode': 'ANSWERED'}
== Sample Q&A 2 ==
{'answer': '[b1] Beta service deployment guide: run deploy.sh --prod to release to production. Always run the test suite first.', 'sources': ['b1'], 'project_id': 'beta', 'mode': 'ANSWERED'}
== Failure demo: cross-project leak attempt ==
{'answer': '[b1] Beta service deployment guide: run deploy.sh --prod to release to production. Always run the test suite first.', 'sources': ['b1'], 'project_id': 'beta', 'mode': 'ANSWERED'}


In [13]:
kbs, acl = build_seed_environment()
bot = Chatbot(kbs, acl)
bot.login("alice")
bot.select_project("alpha")
print(bot.ask("how many annual leaves do employees get?"))
bot.select_project("gamma")
print(bot.ask("what is the refund window in days?"))
demos = demo_conversations()
print("\n5 failure-case demos:", [d["title"] for d in demos])
print("\nUnauthorized-access demo:", demos[1])


{'answer': '[a1] Alpha leave policy: employees receive 20 annual leaves per year. Annual leave accrues monthly and rolls over up to 5 days.', 'sources': ['a1', 'a2'], 'project_id': 'alpha', 'mode': 'ANSWERED'}
{'answer': 'Conflicting evidence in gamma (g1 vs g2). Could you clarify which applies?', 'sources': ['g1', 'g2'], 'project_id': 'gamma', 'mode': 'CONTRADICTION'}

5 failure-case demos: ['Cross-project leak attempt (alpha question while in beta)', 'Unauthorized project access (bob asks alpha)', 'Topic switch mid-conversation (alpha -> gamma)', 'Missing evidence (out-of-scope question)', 'Contradictory documents (30-day vs 7-day refund)']

Unauthorized-access demo: {'title': 'Unauthorized project access (bob asks alpha)', 'turns': [{'question': 'project alpha what is the annual leave policy?', 'mode': 'DENIED', 'sources': [], 'answer': "Access denied: you are not authorized for project 'alpha'."}], 'fix_explanation': 'ACL is checked on every selection and implicit switch; unauthori

## Question 7 — Local-first legal transcription pipeline (EN/HI/Hinglish)
Simulated STT → deterministic correction (negations preserved) → relevance filter → extraction against versioned local DB (never invents sections).

In [14]:
"""Question 7 — Local-first legal transcription pipeline (Hindi / English / Hinglish).

Deterministic, offline, standard-library only:

    simulated STT -> transcript correction -> relevance filtering
    -> legal extraction against a versioned local DB -> benchmark metrics

Guarantees:
  * No model downloads, no network calls, no GPU, no API spend.
  * Negations ("not" / "nahi" / Devanagari "नहीं") are never dropped.
  * Legal sections are never invented: an unknown section is flagged as
    "not in local DB v1.0 — verify manually".
"""

from __future__ import annotations

import re
import time
from typing import Any

# ---------------------------------------------------------------------------
# Relevance labels
# ---------------------------------------------------------------------------
LEGALLY_RELEVANT = "LEGALLY_RELEVANT"
IRRELEVANT = "IRRELEVANT"
UNCERTAIN = "UNCERTAIN"

# ---------------------------------------------------------------------------
# Versioned local legal database — the ONLY source of legal provisions.
# Never invent entries at runtime.
# ---------------------------------------------------------------------------
DB_VERSION = "1.0"
LEGAL_DB: dict[str, dict[str, str]] = {
    "Section 73": {"act": "Indian Contract Act", "note": "compensation for breach"},
    "Section 420": {"act": "IPC (old)", "note": "cheating — verify against BNS mapping"},
    "Section 138": {"act": "Negotiable Instruments Act", "note": "cheque dishonour"},
}

# ---------------------------------------------------------------------------
# Simulated STT fixtures (fixed, in-code, no acoustic model).
# ---------------------------------------------------------------------------
AUDIO_FIXTURES: dict[str, dict[str, Any]] = {
    "en_agreement_01": {
        "language": "en",
        "wer_hint": 0.12,
        "transcript": (
            "Good morning everyone, the weather is pleasant today.\n"
            "Sharma versus Verma signed an agreement on 12 January 2023.\n"
            "The defendant did not pay the amount twlce as agreed.\n"
            "This is a clear breach of the contract.\n"
            "As per section seventy three the claimant can seek compensation.\n"
            "The client also mentioned Section 999 which we must verify.\n"
            "After the meeting we watched a movie."
        ),
    },
    "hi_payment_01": {
        "language": "hinglish",
        "wer_hint": 0.18,
        "transcript": (
            "नमस्ते सर, आज मौसम अच्छा है।\n"
            "कल की बात है कि भुगतान 15 March 2024 तक नहीं हुआ।\n"
            "समझौता के अनुसार धारा 73 लागू हो सकती है।\n"
            "बाद में movie देखने चलेंगे।"
        ),
    },
    "en_chitchat_01": {
        "language": "en",
        "wer_hint": 0.05,
        "transcript": (
            "Hey how are you doing today.\n"
            "The weather is really nice this weekend.\n"
            "I tried a new food place yesterday.\n"
            "Did you watch the cricket match last night.\n"
            "The movie was amazing."
        ),
    },
}


class SimulatedSTT:
    """Deterministic offline stand-in for a speech-to-text model."""

    def __init__(self, fixtures: dict[str, dict[str, Any]] | None = None) -> None:
        self._fixtures = fixtures if fixtures is not None else AUDIO_FIXTURES

    def transcribe(self, audio_id: str) -> dict[str, Any]:
        if audio_id not in self._fixtures:
            raise KeyError(f"unknown audio_id: {audio_id}")
        fixture = self._fixtures[audio_id]
        return {
            "transcript": fixture["transcript"],
            "language": fixture["language"],
            "wer_hint": fixture["wer_hint"],
        }


# ---------------------------------------------------------------------------
# Transcript correction (deterministic; negations preserved by construction)
# ---------------------------------------------------------------------------
_ASR_CORRECTIONS: list[tuple[str, str]] = [
    ("section seventy three", "Section 73"),
    ("section four twenty", "Section 420"),
    ("section one thirty eight", "Section 138"),
    ("dhara seventy three", "Section 73"),
    ("twlce", "twice"),
    ("paymnt", "payment"),
    ("agreemnt", "agreement"),
    ("samjauta", "samjhauta"),
]


def correct_transcript(text: str) -> str:
    """Apply fixed ASR-error replacements and whitespace normalisation.

    Only substitutions are performed; no token is ever removed, so negations
    such as "not", "nahi" and "नहीं" survive untouched.
    """
    out = text
    for wrong, right in _ASR_CORRECTIONS:
        out = re.sub(re.escape(wrong), right, out, flags=re.IGNORECASE)
    out = re.sub(r"[ \t]+", " ", out)
    lines = [line.strip() for line in out.split("\n")]
    return "\n".join(line for line in lines if line)


# ---------------------------------------------------------------------------
# Legal relevance classification (keyword rules, prefix-matched)
# ---------------------------------------------------------------------------
_RELEVANT_ASCII: list[str] = [
    "section", "agreement", "agree", "contract", "payment", "pay", "paid",
    "amount", "breach", "court", "police", "witness", "allegation", "evidence",
    "compensation", "claim", "clause", "provision", "notice", "damages",
    "defendant", "plaintiff", "claimant", "accused", "liability", "dispute",
    "cheque", "dishonour", "fraud", "cheat", "date", "dhara", "samjhauta",
    "bhugtan", "tareekh", "adalat", "dawa", "gawah",
]
_RELEVANT_NONASCII: list[str] = [
    "धारा", "समझौता", "भुगतान", "तारीख", "अदालत", "दावा", "गवाह",
]
_IRRELEVANT_ASCII: list[str] = [
    "weather", "food", "movie", "cricket", "weekend", "song", "lunch",
    "dinner", "match", "how are you", "mausam", "khana", "gaana",
]
_IRRELEVANT_NONASCII: list[str] = ["मौसम", "खाना", "फिल्म", "गाना", "क्रिकेट"]


def _match_ascii(text_lower: str, keywords: list[str]) -> bool:
    return any(re.search(rf"\b{re.escape(kw)}", text_lower) for kw in keywords)


def _match_substr(text_lower: str, keywords: list[str]) -> bool:
    return any(kw in text_lower for kw in keywords)


def classify_segment(segment: str) -> str:
    """Label one segment; legally relevant facts win over small talk."""
    text = segment.lower()
    if _match_ascii(text, _RELEVANT_ASCII) or _match_substr(text, _RELEVANT_NONASCII):
        return LEGALLY_RELEVANT
    if _match_ascii(text, _IRRELEVANT_ASCII) or _match_substr(text, _IRRELEVANT_NONASCII):
        return IRRELEVANT
    return UNCERTAIN


def filter_transcript(
    segments: list[str],
) -> tuple[list[dict[str, str]], list[dict[str, str]]]:
    """Return (kept, dropped): keep relevant + uncertain, drop irrelevant."""
    kept: list[dict[str, str]] = []
    dropped: list[dict[str, str]] = []
    for segment in segments:
        label = classify_segment(segment)
        item = {"text": segment, "label": label}
        (dropped if label == IRRELEVANT else kept).append(item)
    return kept, dropped


# ---------------------------------------------------------------------------
# Legal extraction against the versioned local DB
# ---------------------------------------------------------------------------
_SECTION_RE = re.compile(
    r"\b(?:Section|Dhara|धारा)\s*(\d{1,4}[A-Za-z]?)", re.IGNORECASE
)
_DATE_RE = re.compile(r"\b\d{1,2}\s+[A-Za-z\u0900-\u097F]+\s+\d{4}\b")
_PARTY_RE = re.compile(r"\b([A-Z][a-z]+)\s+(?:vs|versus|and)\s+([A-Z][a-z]+)\b")

_LEGAL_TERMS: list[str] = [
    "agreement", "contract", "breach", "compensation", "damages", "allegation",
    "evidence", "witness", "negligence", "cheque", "dishonour", "fraud",
    "cheating", "court", "police", "clause", "provision", "notice", "claim",
    "defendant", "plaintiff", "accused", "samjhauta", "dhara", "adalat",
    "dawa", "gawah", "समझौता", "धारा", "अदालत", "दावा", "गवाह",
]


def _norm_section(number: str) -> str:
    return f"Section {number.upper()}"


def find_sections(text: str) -> list[str]:
    """Explicitly mentioned sections, normalised to e.g. "Section 73"."""
    found: list[str] = []
    for match in _SECTION_RE.finditer(text):
        key = _norm_section(match.group(1))
        if key not in found:
            found.append(key)
    return found


def lookup_section(section: str) -> dict[str, Any]:
    """DB lookup only. Unknown sections are flagged, never invented."""
    key = section if section.startswith("Section") else _norm_section(section)
    if key in LEGAL_DB:
        record = LEGAL_DB[key]
        return {
            "section": key,
            "known": True,
            "act": record["act"],
            "note": record["note"],
            "status": "STATED",
        }
    return {
        "section": key,
        "known": False,
        "warning": f"not in local DB v{DB_VERSION} — verify manually",
        "status": "STATED",
    }


def _suggest_keys(text: str) -> list[str]:
    """Keyword heuristics returning candidate DB keys (never free text)."""
    low = text.lower()

    def present(*words: str) -> bool:
        return any(word in low for word in words)

    keys: list[str] = []
    if present("payment", "pay", "bhugtan", "भुगतान") and present(
        "breach", "dishonour", "cheque", "bounced", "default"
    ):
        keys.append("Section 73")
    if present("cheque", "dishonour", "bounced") and present(
        "payment", "bhugtan", "भुगतान", "bank"
    ):
        keys.append("Section 138")
    if present("cheating", "fraud", "fraudster"):
        keys.append("Section 420")
    return keys


def extract_legal_info(transcript: str) -> dict[str, Any]:
    """Extract parties, dates, mentioned sections, DB provisions and terms."""
    sections = find_sections(transcript)
    stated = [lookup_section(section) for section in sections]
    stated_keys = {entry["section"] for entry in stated}

    suggested: list[dict[str, Any]] = []
    for key in _suggest_keys(transcript):
        if key in stated_keys:
            continue
        record = LEGAL_DB[key]
        suggested.append(
            {
                "section": key,
                "known": True,
                "act": record["act"],
                "note": record["note"],
                "status": "SUGGESTED",
            }
        )

    parties: list[str] = []
    for match in _PARTY_RE.finditer(transcript):
        party = re.sub(r"\s+", " ", match.group(0)).strip()
        if party not in parties:
            parties.append(party)

    terms: list[str] = []
    low = transcript.lower()
    for term in _LEGAL_TERMS:
        if term.isascii():
            if re.search(rf"\b{re.escape(term)}", low) and term not in terms:
                terms.append(term)
        elif term in low and term not in terms:
            terms.append(term)
    terms.sort()

    return {
        "parties": parties,
        "dates": _DATE_RE.findall(transcript),
        "sections_mentioned": [entry["section"] for entry in stated],
        "suggested_provisions": stated + suggested,
        "legal_terms": terms,
    }


# ---------------------------------------------------------------------------
# End-to-end pipeline
# ---------------------------------------------------------------------------
def split_segments(transcript: str) -> list[str]:
    return [line.strip() for line in transcript.split("\n") if line.strip()]


def run_pipeline(audio_id: str) -> dict[str, Any]:
    """Run STT -> correction -> filtering -> extraction and time the result."""
    start = time.perf_counter()
    raw = SimulatedSTT().transcribe(audio_id)
    corrected = correct_transcript(raw["transcript"])
    segments = split_segments(corrected)
    labeled = [{"text": s, "label": classify_segment(s)} for s in segments]
    kept, _dropped = filter_transcript(segments)
    relevant_only = [item["text"] for item in kept]
    extraction = extract_legal_info(corrected)
    latency_ms = (time.perf_counter() - start) * 1000.0

    return {
        "audio_id": audio_id,
        "language": raw["language"],
        "corrected_transcript": corrected,
        "segments": labeled,
        "relevant_only": relevant_only,
        "extraction": extraction,
        "metrics": {
            "latency_ms": round(latency_ms, 4),
            "wer_proxy": raw["wer_hint"],
            "api_cost_usd": 0.0,
            "cpu_note": "local deterministic rules; no model download, no network, no GPU",
        },
    }


def benchmark() -> list[dict[str, Any]]:
    """Run every fixture, print a table, and return the rows."""
    rows: list[dict[str, Any]] = []
    for audio_id in AUDIO_FIXTURES:
        result = run_pipeline(audio_id)
        segments = result["segments"]
        kept = [seg for seg in segments if seg["label"] != IRRELEVANT]
        ratio = len(kept) / len(segments) if segments else 0.0
        metrics = result["metrics"]
        rows.append(
            {
                "audio_id": audio_id,
                "latency_ms": metrics["latency_ms"],
                "wer_proxy": metrics["wer_proxy"],
                "relevant_ratio": round(ratio, 3),
                "api_cost_usd": metrics["api_cost_usd"],
            }
        )

    print(f"{'audio_id':<18}{'latency_ms':>12}{'wer_proxy':>11}{'rel_ratio':>11}{'cost_usd':>10}")
    for row in rows:
        print(
            f"{row['audio_id']:<18}{row['latency_ms']:>12.4f}"
            f"{row['wer_proxy']:>11.2f}{row['relevant_ratio']:>11.3f}"
            f"{row['api_cost_usd']:>10.2f}"
        )
    return rows


if __name__ == "__main__":
    import json

    benchmark()
    print("\nFull pipeline (en_agreement_01) — offline, cost $0.00:")
    print(json.dumps(run_pipeline("en_agreement_01"), ensure_ascii=False, indent=2))


audio_id            latency_ms  wer_proxy  rel_ratio  cost_usd
en_agreement_01         1.3173       0.12      0.714      0.00
hi_payment_01           0.6649       0.18      0.500      0.00
en_chitchat_01          0.6356       0.05      0.000      0.00

Full pipeline (en_agreement_01) — offline, cost $0.00:
{
  "audio_id": "en_agreement_01",
  "language": "en",
  "corrected_transcript": "Good morning everyone, the weather is pleasant today.\nSharma versus Verma signed an agreement on 12 January 2023.\nThe defendant did not pay the amount twice as agreed.\nThis is a clear breach of the contract.\nAs per Section 73 the claimant can seek compensation.\nThe client also mentioned Section 999 which we must verify.\nAfter the meeting we watched a movie.",
  "segments": [
    {
      "text": "Good morning everyone, the weather is pleasant today.",
      "label": "IRRELEVANT"
    },
    {
      "text": "Sharma versus Verma signed an agreement on 12 January 2023.",
      "label": "LEGALLY_RELEVAN

In [15]:
import json
rows = benchmark()
print("\nFull pipeline (en_agreement_01):")
res = run_pipeline("en_agreement_01")
print("language:", res["language"])
print("sections:", res["extraction"]["sections_mentioned"])
print("dates:", res["extraction"]["dates"])
print("metrics:", res["metrics"])
print(json.dumps(res["extraction"]["suggested_provisions"], indent=2))


audio_id            latency_ms  wer_proxy  rel_ratio  cost_usd
en_agreement_01         0.5111       0.12      0.714      0.00
hi_payment_01           0.5305       0.18      0.500      0.00
en_chitchat_01          0.5657       0.05      0.000      0.00

Full pipeline (en_agreement_01):
language: en
sections: ['Section 73', 'Section 999']
dates: ['12 January 2023']
metrics: {'latency_ms': 0.4544, 'wer_proxy': 0.12, 'api_cost_usd': 0.0, 'cpu_note': 'local deterministic rules; no model download, no network, no GPU'}
[
  {
    "section": "Section 73",
    "known": true,
    "act": "Indian Contract Act",
    "note": "compensation for breach",
    "status": "STATED"
  },
  {
    "section": "Section 999",
    "known": false,
    "warning": "not in local DB v1.0 \u2014 verify manually",
    "status": "STATED"
  }
]


## Verification — inline assertions (mirrors `tests/`, fully self-contained)

In [16]:
# Q1
r = process_documents(documents)
assert r["counts"] == {"ACCEPT": 3, "REVIEW": 1, "RETAKE": 1}, r["counts"]
assert round(r["average_quality"], 3) == 0.734
assert r["duplicates_removed"] == 2

# Q2
assert check_image_quality(make_clear_image())["is_usable"] is True
assert check_image_quality(make_blurred_image())["is_usable"] is False
assert check_image_quality(make_dark_image())["is_usable"] is False

# Q3
assert classify_confidence(0.95) == "HIGH_CONFIDENCE"
assert classify_confidence(0.60) == "REVIEW_REQUIRED"
assert classify_confidence(0.59) == "LOW_CONFIDENCE"
assert clean_text("  hello   world ") == "hello world"
s = []
out = review_ocr_results(OCR_FIXTURE, corrections=s)
assert s and s[0].original != "" and s[0].corrected
assert validate_medicine("Amoxicillin")["method"] == "exact"
assert validate_medicine("Metfromin")["method"] == "fuzzy"

# Q4
assert build_prescription_json(valid)["validation_status"] == "VALID"
assert build_prescription_json(missing_med)["validation_status"] == "INVALID"
assert build_prescription_json(swapped)["validation_status"] == "REVIEW_REQUIRED"
assert build_prescription_json(bad_dur)["validation_status"] == "REVIEW_REQUIRED"

# Q5 (compact re-run)
import cv2 as _cv2, numpy as _np
_pipe = TextExtractionPipeline()
_rx, _m = _pipe.run(make_rx(), expected_text=EXPECTED)
assert _rx["validation_status"] == "VALID" and _m["llm_calls"] == 1
_bad = _cv2.convertScaleAbs(_cv2.GaussianBlur(make_rx(11), (41, 41), 0), alpha=0.3, beta=0)
_rb, _mb = _pipe.run(_bad)
assert _rb["validation_status"] == "NEEDS_RETAKE" and _mb["llm_calls"] == 0
_, _m3 = _pipe.run(make_rx(), expected_text=EXPECTED)
assert _m3["cached"] is True and _m3["llm_calls"] == 0

# Q6
_kbs, _acl = build_seed_environment()
_b = Chatbot(_kbs, _acl); _b.login("alice"); _b.select_project("alpha")
_resp = _b.ask("how many annual leaves do employees get?")
assert _resp["mode"] == ANSWERED and "a1" in _resp["sources"]
_b.select_project("gamma")
assert _b.ask("what is the refund window in days?")["mode"] == CONTRADICTION
_bob = Chatbot(_kbs, _acl); _bob.login("bob")
try:
    _bob.select_project("alpha")
    raise SystemExit("bob should be denied alpha")
except PermissionError:
    pass
_bob.select_project("beta")
assert _bob.ask("project alpha what is the annual leave policy?")["mode"] == DENIED

# Q7
_r7 = run_pipeline("en_agreement_01")
assert "Section 73" in _r7["extraction"]["sections_mentioned"]
_unknown = [p for p in _r7["extraction"]["suggested_provisions"] if p["section"] == "Section 999"]
assert _unknown and _unknown[0]["known"] is False
assert all(row["api_cost_usd"] == 0.0 for row in benchmark())

print("ALL INLINE CHECKS PASSED: Q1 Q2 Q3 Q4 Q5 Q6 Q7")


audio_id            latency_ms  wer_proxy  rel_ratio  cost_usd
en_agreement_01         0.4061       0.12      0.714      0.00
hi_payment_01           0.5252       0.18      0.500      0.00
en_chitchat_01          0.5793       0.05      0.000      0.00
ALL INLINE CHECKS PASSED: Q1 Q2 Q3 Q4 Q5 Q6 Q7


## AI-usage disclosure
OpenCode (agentic coding assistant, model: Muse Spark) was used as the coding agent to write the code fast — implementation speed only. The logic, approach, and design decisions for every question were given by me personally. Usage level: **Medium**. No other external assistance.